# Notebook 04 — Test the Deployed Serverless Endpoint

**Module:** ITI113 Machine Learning & Operations
**Focus Area:** C — MLOps & Deployment
**Estimated Runtime:** 2-5 minutes (endpoint must already be deployed by Notebook 03)

---

## What this notebook does

1. Finds and confirms team03's deployed SageMaker Serverless Endpoint is active.
2. Inspects the deployed model artifact, so the endpoint can be traced back to the SageMaker Model Registry package it was approved from.
3. Tests the endpoint with `boto3` using the exact request/response shape the Streamlit app uses: `{"text": "..."}` in, `{"prediction", "label", "probability"}` out.
4. Tests batch invocation (`{"instances": [...]}`) with several messages at once.
5. A troubleshooting reference and an optional (commented-out) cleanup cell.

This notebook does **not** build a UI. Unlike the tutor's `04_optional_gradio_serverless_endpoint_demo.ipynb`, this project's actual demo app is Streamlit (`pages/1_Scam_Detector.py`), which runs as its own process via `streamlit run app.py` rather than launching inline in a notebook cell the way Gradio does. This notebook's job is to validate the endpoint and settle on the exact `invoke_endpoint` call the Streamlit app will use — that wiring itself happens directly in `pages/1_Scam_Detector.py`, not here.

> **Adapted for team03 (Ong Hui Lin, Student 2 — MLOps & Deployment) from the ITI113 course template notebook `04_optional_gradio_serverless_endpoint_demo.ipynb`.** The endpoint-discovery, activation-check and model-artifact-inspection logic (Sections 1–2) follow the tutor's original, generalised from a hardcoded endpoint name to searching for team03's own. `invoke_heart_endpoint()` becomes `invoke_scam_detector()`, sending `{"text": "..."}` instead of 13 numeric fields, to match Notebook 03's `inference.py`. The tutor's Gradio UI sections are not applicable here — this project's demo client is a standalone Streamlit app.

In [1]:
# After running, restart the kernel before continuing if packages were upgraded.
%pip install --upgrade boto3 botocore

Note: you may need to restart the kernel to use updated packages.


## 0. Configuration and Endpoint Discovery

Searches for serverless endpoints created by this team (matching `TEAM_ID` in the endpoint name), the same way the tutor's notebook does, rather than hardcoding the endpoint name.

In [2]:
import boto3
import json

REGION = "ap-southeast-1"
TEAM_ID = "team03"
STUDENT_ID = "s301"
PROJECT_NAME = "crypto-scam-detector"

# Matches the naming used in Notebook 03.
ENDPOINT_NAME = f"iti113-{TEAM_ID}-{PROJECT_NAME}"

sts = boto3.client("sts", region_name=REGION)
sm = boto3.client("sagemaker", region_name=REGION)
runtime = boto3.client("sagemaker-runtime", region_name=REGION)

print("Region:", REGION)
print("Team ID:", TEAM_ID)
print("Student ID:", STUDENT_ID)
print("Expected endpoint name:", ENDPOINT_NAME)
print("AWS identity:", sts.get_caller_identity()["Arn"])

response = sm.list_endpoints(
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=100
)

team_endpoints = [
    ep for ep in response["Endpoints"]
    if TEAM_ID in ep["EndpointName"].lower()
]

print(f"\nEndpoints found for {TEAM_ID}: {len(team_endpoints)}")
for ep in team_endpoints:
    print(f"  {ep['EndpointName']}  (status: {ep['EndpointStatus']})")

Region: ap-southeast-1
Team ID: team03
Student ID: s301
Expected endpoint name: iti113-team03-crypto-scam-detector
AWS identity: arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team03/SageMaker

Endpoints found for team03: 1
  iti113-team03-crypto-scam-detector  (status: InService)


## 1. Confirm the Serverless Endpoint Is Active

The endpoint must show `InService` before it can be invoked.

In [3]:
try:
    for ep in team_endpoints:
        ENDPOINT_NAME = ep["EndpointName"]
        endpoint_desc = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
        print("\nEndpoint name:", endpoint_desc["EndpointName"])
        print("Status:", endpoint_desc["EndpointStatus"])
        print("Creation time:", endpoint_desc["CreationTime"])
        print("Last modified:", endpoint_desc["LastModifiedTime"])

    if not team_endpoints:
        print(
            f"No endpoints found for {TEAM_ID}. Run Notebook 03's deploy section first, "
            "then rerun the discovery cell above."
        )
except Exception as e:
    print("Unable to describe endpoint.")
    print("Check that the endpoint exists and that your role has permission to access it.")
    print("Error:", e)


Endpoint name: iti113-team03-crypto-scam-detector
Status: InService
Creation time: 2026-08-01 06:51:01.159000+00:00
Last modified: 2026-08-01 06:53:39.359000+00:00


## 2. Inspect the Endpoint's Model Artifact (Optional)

Traces the deployed endpoint back to the SageMaker Model Registry package it was approved from -- useful for the Final Report's traceability discussion (dataset → preprocessing → training job → model registry → endpoint).

In [4]:
for ep in team_endpoints:
    ENDPOINT_NAME = ep["EndpointName"]

    print("=" * 100)
    print("Endpoint name:", ENDPOINT_NAME)

    try:
        endpoint_desc = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
        endpoint_config_name = endpoint_desc["EndpointConfigName"]

        print("Endpoint status:", endpoint_desc["EndpointStatus"])
        print("Endpoint config:", endpoint_config_name)

        endpoint_config = sm.describe_endpoint_config(EndpointConfigName=endpoint_config_name)
        production_variants = endpoint_config.get("ProductionVariants", [])

        if not production_variants:
            print("No production variants found.")
            continue

        for variant in production_variants:
            print("-" * 80)
            variant_name = variant.get("VariantName")
            model_name = variant.get("ModelName")

            print("Variant name:", variant_name)
            print("Model name:", model_name)

            model_desc = sm.describe_model(ModelName=model_name)
            containers = model_desc.get("Containers") or model_desc.get("PrimaryContainer")

            if isinstance(containers, dict):
                containers = [containers]

            for container in containers or []:
                model_package_arn = container.get("ModelPackageName")
                if model_package_arn:
                    print("Model package:", model_package_arn)
                    package_desc = sm.describe_model_package(ModelPackageName=model_package_arn)
                    print("Model package status:", package_desc.get("ModelApprovalStatus"))
                else:
                    print("Model artifact (ModelDataUrl):", container.get("ModelDataUrl"))

    except Exception as e:
        print("Could not inspect this endpoint's model chain.")
        print("Error:", e)

Endpoint name: iti113-team03-crypto-scam-detector
Endpoint status: InService
Endpoint config: iti113-team03-crypto-scam-detector


--------------------------------------------------------------------------------
Variant name: AllTraffic
Model name: team03-CryptoScamDetector-2026-08-01-06-50-59-528


Model package: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team03-CryptoScamDetector/1
Model package status: Approved


## 3. Test Endpoint Invocation with boto3

The endpoint accepts JSON input with the raw message text -- the same shape `pages/1_Scam_Detector.py` will send:

```json
{"text": "your message here"}
```

and returns:

```json
[{"prediction": 1, "label": "Scam", "probability": 0.87}]
```

`inference.py` (from Notebook 03) performs the cleaning, engineered-feature extraction, and TF-IDF transform internally -- callers only ever send raw text.

In [5]:
import json

runtime = boto3.client("sagemaker-runtime", region_name=REGION)


def invoke_scam_detector(text, endpoint_name=ENDPOINT_NAME):
    """
    Invoke the deployed crypto-scam-detector endpoint with a single raw message.

    This is the exact call pages/1_Scam_Detector.py should make once wired up --
    validate it here first, then reuse it in the Streamlit app.
    """
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps({"text": text}),
    )
    return json.loads(response["Body"].read())[0]


scam_message = (
    "URGENT: Your wallet has been selected for a guaranteed 100% profit airdrop! "
    "Deposit 500 USDT to your wallet address within 1 hour to claim now. "
    "Contact us on Telegram immediately, don't miss out!"
)

result = invoke_scam_detector(scam_message)
print("SCAM-STYLE MESSAGE")
print(f"  Prediction  : {result['label']}")
print(f"  Probability : {result['probability']:.1%}")

SCAM-STYLE MESSAGE
  Prediction  : Scam
  Probability : 75.8%


In [6]:
legit_message = (
    "Been dollar-cost averaging into ETH for about a year now, curious what "
    "everyone's thoughts are on the current market conditions."
)

borderline_message = (
    "Hey, our community wallet is doing a small giveaway this week, check the "
    "pinned post in the group for details."
)

for label, msg in [("LEGIT-STYLE MESSAGE", legit_message), ("BORDERLINE / AMBIGUOUS MESSAGE", borderline_message)]:
    result = invoke_scam_detector(msg)
    print(label)
    print(f"  Prediction  : {result['label']}")
    print(f"  Probability : {result['probability']:.1%}")
    print()

LEGIT-STYLE MESSAGE
  Prediction  : Legitimate
  Probability : 21.8%



BORDERLINE / AMBIGUOUS MESSAGE
  Prediction  : Legitimate
  Probability : 32.7%



## 4. Batch Invocation

`inference.py` also accepts `{"instances": [...]}` for multiple messages in a single request -- useful for testing several examples at once, or if the app ever needs to score a batch.

In [7]:
batch_messages = [
    scam_message,
    legit_message,
    borderline_message,
    "Congratulations! You have been selected to receive a free NFT, claim your prize now before it expires!",
    "Anyone else having trouble syncing their hardware wallet after the latest firmware update?",
]

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps({"instances": [{"text": t} for t in batch_messages]}),
)

results = json.loads(response["Body"].read())

print(f"{'Message (truncated)':<70} {'Label':<12} Probability")
print("-" * 100)
for msg, res in zip(batch_messages, results):
    truncated = (msg[:65] + "...") if len(msg) > 65 else msg
    print(f"{truncated:<70} {res['label']:<12} {res['probability']:.1%}")

Message (truncated)                                                    Label        Probability
----------------------------------------------------------------------------------------------------
URGENT: Your wallet has been selected for a guaranteed 100% profi...   Scam         75.8%
Been dollar-cost averaging into ETH for about a year now, curious...   Legitimate   21.8%
Hey, our community wallet is doing a small giveaway this week, ch...   Legitimate   32.7%
Congratulations! You have been selected to receive a free NFT, cl...   Legitimate   44.8%
Anyone else having trouble syncing their hardware wallet after th...   Legitimate   21.1%


## 5. Troubleshooting

| Problem | Possible Cause | What to Check |
|---|---|---|
| `ValidationException: Could not find endpoint` | Wrong or not-yet-deployed endpoint name | Run the discovery cell in Section 0 again; confirm Notebook 03's deploy section completed. |
| `AccessDeniedException` | Wrong team role, or endpoint outside team permission | Confirm you're using your own SageMaker Studio profile and your own team's endpoint. |
| `ModelError` | Payload doesn't match `inference.py`'s expected shape | Confirm the request is `{"text": "..."}` or `{"instances": [{"text": "..."}, ...]}`, not raw numeric fields. |
| Endpoint charges continue | Endpoint still exists | Delete the endpoint after testing if it's no longer needed (Section 6). |

## 6. Optional Cleanup Reminder

Do **not** run cleanup if the endpoint is still needed for the Streamlit app, further testing, or your Final Presentation demo.

When truly finished, delete the endpoint to avoid leaving unused resources active.

In [8]:
# Optional cleanup example. Uncomment only when you really want to delete the endpoint.

# endpoint_desc = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
# endpoint_config_name = endpoint_desc["EndpointConfigName"]

# sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
# print(f"Deleted endpoint: {ENDPOINT_NAME}")

# sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
# print(f"Deleted endpoint config: {endpoint_config_name}")

---
## Next Step

`invoke_scam_detector()` above is the exact call to reuse in `pages/1_Scam_Detector.py`, replacing the current rule-based placeholder logic: send `{"text": message}` to `ENDPOINT_NAME` via `boto3` `sagemaker-runtime.invoke_endpoint()`, and use the returned `label`/`probability` in place of the placeholder `risk_level`/`scam_probability` values. `requirements.txt` will need `boto3` added.